# TextSplitter的源码剖析
## 主要讲一下内部的三个方法：
### 方法1： split_text(self, text: str) -> list[str]:

传入的参数类型：文本内容（或字符串），返回值类型：字符串列表

此方法是抽象方法，具体的实现细节由子类来决定

### 方法2： create_documents(self, texts: list[str],...) -> list[Document]:

传入的参数类型：字符串列表，返回值类型：Document对象列表

此方法的底层调用了split_text()，即将参数中的每一个字符串都传入split_text()中执行，得到的字符串列表中，将字符串封装为Document对象，就构成了list[Document]。

### 方法3： split_documents(self, documents: Iterable[Document]) -> list[Document]:

传入的参数类型：Document对象列表，返回值类型：Document对象列表

此方法的底层调用了create_documents(),将参数中的每一个Document对象，提取其page_content字段，则构成了字符串列表，然后调用方法2即可。



### 这一节课学习：字符串文本的分割器
CharacterTextSplitter：Split by character
#### 举例1：分隔符为空""
#### 常用参数
##### chunk_size,#块大小
##### chunk_overlap,# 块与块之间的重复字符数
##### separator，# 覆盖器
##### length_function, #长度函数
### separator优先原则：
#### 当设置了 separator（如"。"），分割器会首先尝试在分隔符处分割，然后再考虑 chunk_size。这是为了避免在句子中间硬性切断。这种设计是为了：

#### 优先保持语义完整性（不切断句子）
#### 避免产生无意义的碎片（如半个单词/不完整句子）
#### 如果 chunk_size 比片段小，无法拆分片段，导致 overlap失效。
#### chunk_overlap仅在合并后的片段之间生效（如果 chunk_size 足够大）。如果没有合并的片段，则 overlap失效。

In [1]:
from langchain_text_splitters import CharacterTextSplitter

# 1.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。
"""

# 2.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=50,#块大小
    chunk_overlap=5,# 块与块之间的重复字符数
    separator="" # 设置为空字符串时，表示禁用分隔符优先
)

# 3.分割文本
texts = splitter.split_text(text)
# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(chunk)
    print("-" * 50)

块 1:长度：49
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发
--------------------------------------------------
块 2:长度：22
象，使开发者能够更容易地构建复杂的应用程序。
--------------------------------------------------


### 举例2，指定分隔符

In [2]:
from langchain_text_splitters import CharacterTextSplitter

# 1.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。
"""

# 2.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=50,#块大小
    chunk_overlap=5,# 块与块之间的重复字符数
    separator="。" # 用中文句号作为分隔符
)

# 3.分割文本
texts = splitter.split_text(text)
# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(chunk)
    print("-" * 50)

块 1:长度：33
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的
--------------------------------------------------
块 2:长度：32
它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。
--------------------------------------------------


### 举例3：指定分割符，并且保留分隔符


In [3]:
from langchain_text_splitters import CharacterTextSplitter

# 1.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。
"""

# 2.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=50,#块大小
    chunk_overlap=5,# 块与块之间的重复字符数
    separator="。", # 用中文句号作为分隔符
    keep_separator=True
)

# 3.分割文本
texts = splitter.split_text(text)
# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(chunk)
    print("-" * 50)

块 1:长度：33
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的
--------------------------------------------------
块 2:长度：33
。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。
--------------------------------------------------


### 举例4，两个块的长度加起来还比chuck_size，小，他会发生合并现象


In [1]:
from langchain_text_splitters import CharacterTextSplitter

# 1.示例文本
text = """
text = "这是第一段文本。这是第二段内容。最后一段结束。"
"""

# 2.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=20,#块大小
    chunk_overlap=8,# 块与块之间的重复字符数
    separator="。", # 用中文句号作为分隔符
    keep_separator=True
)

# 3.分割文本
texts = splitter.split_text(text)
# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(chunk)
    print("-" * 50)


块 1:长度：15
text = "这是第一段文本
--------------------------------------------------
块 2:长度：17
。这是第二段内容。最后一段结束。"
--------------------------------------------------
